# Project 1: R&D Quintiles (CAPM + FF3)

Based on `recoded_project_0.ipynb`. **R&D:** No R&D vs. five **equally populated quintiles** of firms with XRD; **normalized** only (XRD as % of lagged market equity, same ME as used for value-weighting). **Models:** CAPM and Fama–French 3-factor. Same metrics; no investability flag.

In [2]:
import wrds
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
import plotly.graph_objects as go
from pathlib import Path
import pickle

def connect_wharton(username="zkg232"):
    """Connect to WRDS PostgreSQL (our prior method: explicit username, uses .pgpass for password)."""
    return wrds.Connection(wrds_username=username)

db = connect_wharton()

# Cache for WRDS data to avoid re-running queries
CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)

def load_or_fetch(name, loader):
    path = CACHE_DIR / f"{name}.pkl"
    if path.exists():
        with open(path, 'rb') as f:
            return pickle.load(f)
    data = loader()
    with open(path, 'wb') as f:
        pickle.dump(data, f)
    return data

Loading library list...
Done


## Compustat Query - Get R&D Data and Link to CRSP

In [3]:
comp_sql = """
SELECT
    a.gvkey,
    a.datadate,
    a.xrd,
    a.ni,
    b.lpermno AS permno,
    LEFT(CAST(n.gind AS VARCHAR), 2) AS gsector
FROM
    comp.funda a
JOIN
    crsp.ccmxpf_linktable b ON a.gvkey = b.gvkey
LEFT JOIN
    comp.names n ON a.gvkey = n.gvkey
WHERE
    a.indfmt = 'INDL'
    AND a.datafmt = 'STD'
    AND a.popsrc = 'D'
    AND a.consol = 'C'
    AND a.fyear BETWEEN 1978 AND 2022
    AND a.fic = 'USA'
    AND a.curcd = 'USD'
    AND a.exchg BETWEEN 11 AND 19
    AND (a.sich < 6000 OR a.sich > 6999 OR a.sich IS NULL)
    AND (a.sich != 2834 OR a.sich IS NULL)
    AND b.linktype IN ('LU', 'LC')
    AND b.linkprim IN ('P', 'C')
    AND b.usedflag = 1
    AND (b.linkdt <= a.datadate OR b.linkdt IS NULL)
    AND (b.linkenddt >= a.datadate OR b.linkenddt IS NULL)
"""
comp = load_or_fetch('comp_quintiles_gics', lambda: db.raw_sql(comp_sql))

## CRSP Query - Monthly Returns with Delisting Adjustment

In [4]:
crsp_sql = """
SELECT 
    a.permno,
    a.date,
    a.ret,
    a.prc,
    a.shrout,
    c.dlret
FROM 
    crsp.msf a
INNER JOIN crsp.msenames b ON a.permno = b.permno
    AND a.date BETWEEN b.namedt AND b.nameendt
LEFT JOIN crsp.msedelist c ON a.permno = c.permno
    AND date_trunc('month', a.date) = date_trunc('month', c.dlstdt)
WHERE 
    a.date BETWEEN '1970-01-01' AND '2023-06-30'
    -- AND a.ret IS NOT NULL
    AND a.ret >= -1
    AND b.shrcd IN (10, 11)
    AND (b.siccd < 6000 OR b.siccd > 6999 OR b.siccd IS NULL)
    AND (b.siccd != 2834 OR b.siccd IS NULL)
    AND b.exchcd IN (1, 2, 3)
"""
crsp = load_or_fetch('crsp', lambda: db.raw_sql(crsp_sql))

## Adjust for Delisting Returns

In [5]:
crsp['dlret'] = pd.to_numeric(crsp['dlret'], errors='coerce')
crsp['ret'] = np.where(crsp['dlret'].notna(),
                       (1 + crsp['ret']) * (1 + crsp['dlret']) - 1,
                       crsp['ret'])

## Market Return and Risk-Free Rate

In [6]:
mkt = load_or_fetch('mkt', lambda: db.raw_sql("""
    SELECT date, vwretd, ewretd
    FROM crsp.msi
    WHERE date BETWEEN '1970-01-01' AND '2023-06-30'
"""))

ff = load_or_fetch('ff_factors', lambda: db.raw_sql("""
    SELECT date, rf, mktrf, smb, hml
    FROM ff.factors_monthly
    WHERE date BETWEEN '1970-01-01' AND '2023-06-30'
"""))

## R&D Measure: Lag and (Option) Normalize

In [7]:
comp = comp.assign(
    xrd=pd.to_numeric(comp['xrd'], errors='coerce'),
    ni=pd.to_numeric(comp['ni'], errors='coerce')
).sort_values(['permno', 'datadate'])
for lag in range(1, 5):
    comp[f'xrd_lag{lag}'] = comp.groupby('permno')['xrd'].shift(lag)
comp['xrdw'] = (comp['xrd'].fillna(0) + 0.8*comp['xrd_lag1'].fillna(0) + 0.6*comp['xrd_lag2'].fillna(0)
                + 0.4*comp['xrd_lag3'].fillna(0) + 0.2*comp['xrd_lag4'].fillna(0))
comp['datadate'] = pd.to_datetime(comp['datadate'])
comp['available_date'] = (comp['datadate'] + pd.Timedelta(days=120)).dt.to_period('M').dt.to_timestamp()
comp['gic_sectors'] = comp['gsector'].astype(str).str[:2].replace('', np.nan)
comp = comp.drop_duplicates(subset=['permno', 'datadate'], keep='last')
crsp['date'] = pd.to_datetime(crsp['date']).dt.to_period('M').dt.to_timestamp()

## 4-month (120-day) reporting lag — available_date

In [8]:
pass

## Market Cap - Use Lagged for Value Weighting

In [9]:
crsp = crsp.assign(
    ret=pd.to_numeric(crsp['ret'], errors='coerce'),
    prc=pd.to_numeric(crsp['prc'], errors='coerce'),
    shrout=pd.to_numeric(crsp['shrout'], errors='coerce')
).assign(mktcap=lambda x: x['prc'].abs() * x['shrout'] * 1000).sort_values(['permno', 'date'])
crsp['mktcap_lag'] = crsp.groupby('permno')['mktcap'].shift(1)

## Merge Datasets (monthly formation: reprex-style merge + date filter + keep last)

In [10]:
# Configuration: Date Ranges
START_DATE = '1980-01-01'  # Full sample start date
END_DATE = '2022-12-31'    # Full sample end date
SPLIT_DATE = '2000-01-01'  # Date to split sample into two subperiods

comp_merge = (comp[['permno', 'available_date', 'xrdw', 'gic_sectors']]
    .assign(permno=lambda d: pd.to_numeric(d['permno'], errors='coerce').astype('int64'))
    .dropna(subset=['permno']))
crsp_dates = crsp[['permno', 'date']].dropna().drop_duplicates()
crsp_dates['permno'] = pd.to_numeric(crsp_dates['permno'], errors='coerce').astype('int64')

# 1. Define the validity window: Data expires 12 months after it becomes public
# Ensure 'available_date' is datetime format before this step
comp_merge['valid_end_date'] = comp_merge['available_date'] + pd.DateOffset(months=12)
merged = (crsp_dates.merge(comp_merge, on='permno', how='inner')
    # 1. date >= available_date: Ensures no look-ahead bias
    # 2. date < valid_end_date: Stops using data older than 12 months (stale data)
    .query('(date >= available_date) & (date < valid_end_date)')
    # Sort by available_date so 'keep=last' picks the most recent filing if periods overlap
    .sort_values(['permno', 'date', 'available_date'])
    .drop_duplicates(subset=['permno', 'date'], keep='last')
    .merge(crsp[['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag']], on=['permno', 'date'], how='inner'))

# merged = (crsp_dates.merge(comp_merge, on='permno', how='inner')
#     .query('date >= available_date')
#     .sort_values(['permno', 'date', 'available_date'])
#     .drop_duplicates(subset=['permno', 'date'], keep='last')
#     .merge(crsp[['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag']], on=['permno', 'date'], how='inner'))
merged = merged[(merged['available_date'].dt.year >= int(START_DATE[:4])) & (merged['available_date'].dt.year <= int(END_DATE[:4]))]
merged = merged.dropna(subset=['ret', 'mktcap_lag']).query('mktcap_lag > 0')
# XRDW % ME uses lagged market equity (mktcap_lag), matching the same variable used for value-weighting portfolio returns
merged['xrdw_pct_me'] = np.where(merged['mktcap_lag'].notna() & (merged['mktcap_lag'] > 0), merged['xrdw'] / merged['mktcap_lag'], np.nan)

def gics_mapping():
    """GICS sector code to name (from reprex_final_extended_tuned)."""
    return pd.DataFrame({
        'gic_sectors': ['10', '15', '20', '25', '30', '35', '40', '45', '50', '55', '60'],
        'sector_name': [
            'Energy', 'Materials', 'Industrials', 'Consumer Discretionary',
            'Consumer Staples', 'Health Care', 'Financials',
            'Information Technology', 'Communication Services', 'Utilities', 'Real Estate'
        ]
    })
merged = merged.merge(gics_mapping(), on='gic_sectors', how='left')
merged['sector'] = merged['sector_name']

## Analysis Data: R&D Quintiles (No R&D + Q1–Q5, equal count)

In [11]:

def assign_quintiles_per_date(df):
    out = []
    for date, g in df.groupby('date'):
        no_rd = g[~g['has_rd']].copy()
        no_rd['rd_quintile'] = 'No R&D'
        with_rd = g[g['has_rd']].copy()
        if len(with_rd) < 5:
            with_rd['rd_quintile'] = 'Q1'
        else:
            with_rd['rd_quintile'] = pd.qcut(with_rd['rd_measure'].rank(method='first'), 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5']).astype(str)
        out.append(pd.concat([no_rd, with_rd], ignore_index=True))
    return pd.concat(out, ignore_index=True)

def build_analysis_data(merged):
    """Normalized analysis only: rd_measure = XRDW / lagged ME (same ME used for value-weighting). Includes sector for contribution breakdown."""
    cols = ['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag', 'xrdw', 'xrdw_pct_me']
    if 'sector' in merged.columns:
        cols = cols + ['sector']
    ad = merged[[c for c in cols if c in merged.columns]].copy()
    ad = ad.dropna(subset=['ret'])
    ad['rd_measure'] = ad['xrdw_pct_me']
    ad['has_rd'] = ad['rd_measure'].notna() & (ad['rd_measure'] > 0)
    ad = assign_quintiles_per_date(ad.sort_values(['date', 'permno']))
    ad['portfolio_id'] = ad['rd_quintile']
    return ad

analysis_data_norm = build_analysis_data(merged)


In [23]:
analysis_data_norm.to_csv('analysis_data_norm.csv')



## Portfolio Returns, CAPM, FF3, Tables, Chart

In [13]:

def format_numeric_cols(data, rounding_map):
    data = data.copy()
    for col, digits in rounding_map.items():
        if col in data.columns and pd.api.types.is_numeric_dtype(data[col]):
            data[col] = data[col].round(digits)
    return data

def calculate_portfolio_returns(data, group_var='research_not'):
    ret_col = 'ret' if 'ret' in data.columns else 'ret_crsp'
    prc_col = 'prc' if 'prc' in data.columns else 'prc_crsp'
    data = data[data[ret_col].notna() & data[prc_col].notna() & data['mktcap'].notna()].copy()
    data = data.sort_values(['permno', 'date'])
    if 'mktcap_lag' not in data.columns:
        data['mktcap_lag'] = data.groupby('permno')['mktcap'].shift(1)
    data['prc_lag'] = data.groupby('permno')[prc_col].shift(1)
    data = data[data['mktcap_lag'].notna() & data['prc_lag'].notna()]
    data['date'] = data['date'] + pd.offsets.MonthEnd(0)
    def calc_vw_return(group):
        total_mktcap = group['mktcap_lag'].sum()
        if total_mktcap == 0:
            return np.nan
        weights = group['mktcap_lag'] / total_mktcap
        return (weights * group[ret_col]).sum()
    portfolio_returns = data.groupby(['date', group_var]).apply(
        lambda x: pd.Series({
            'vw_return': calc_vw_return(x),
            'ew_return': x[ret_col].mean(),
            'total_mktcap': x['mktcap_lag'].sum(),
            'n_stocks': len(x)
        })
    ).reset_index()
    return portfolio_returns.sort_values([group_var, 'date'])

def add_long_short_spreads(portfolio_returns, group_col='portfolio_id'):
    if group_col not in portfolio_returns.columns or 'ew_return' not in portfolio_returns.columns:
        return portfolio_returns
    wide_ew = portfolio_returns.pivot(index='date', columns=group_col, values='ew_return')
    wide_vw = portfolio_returns.pivot(index='date', columns=group_col, values='vw_return')
    needed = ['No R&D', 'Q1', 'Q5']
    for col in needed:
        if col not in wide_ew.columns or col not in wide_vw.columns:
            return portfolio_returns
    out = [portfolio_returns]
    for name, minus_col in [('Q5-No R&D', 'No R&D'), ('Q5-Q1', 'Q1')]:
        spread_ew = (wide_ew['Q5'] - wide_ew[minus_col]).dropna()
        spread_vw = (wide_vw['Q5'] - wide_vw[minus_col]).dropna()
        common_idx = spread_ew.index.intersection(spread_vw.index)
        spread_ew, spread_vw = spread_ew.loc[common_idx], spread_vw.loc[common_idx]
        spread_df = pd.DataFrame({
            'date': spread_ew.index,
            group_col: name,
            'ew_return': spread_ew.values,
            'vw_return': spread_vw.values,
            'total_mktcap': 0.0,
            'n_stocks': 0
        })
        out.append(spread_df)
    return pd.concat(out, ignore_index=True).sort_values([group_col, 'date'])

def run_capm_regression(portfolio_returns, market_returns, rf_rate, use_ew_benchmark=False):
    if 'rf' not in market_returns.columns or market_returns['rf'].isna().all():
        raise ValueError("Risk-free rate 'rf' must be present in market_returns (from Fama-French ff.factors_monthly).")
    market_col = 'ewretd' if use_ew_benchmark else 'vwretd'
    portfolio_returns = portfolio_returns.copy()
    portfolio_returns['date'] = portfolio_returns['date'] + pd.offsets.MonthEnd(0)
    cols = ['date', market_col, 'rf']
    market_data = market_returns[cols].copy()
    market_data['date'] = market_data['date'] + pd.offsets.MonthEnd(0)
    market_data = market_data.rename(columns={market_col: 'market_return'})
    reg_data = portfolio_returns.merge(market_data, on='date', how='left')
    reg_data = reg_data[reg_data['portfolio_return'].notna() & reg_data['market_return'].notna() & reg_data['rf'].notna()].copy()
    r = reg_data['rf'].astype(float)
    reg_data['portfolio_excess'] = reg_data['portfolio_return'] - r
    reg_data['market_excess'] = reg_data['market_return'] - r
    reg_data = reg_data[reg_data['portfolio_excess'].notna() & reg_data['market_excess'].notna()].copy()
    if len(reg_data) < 2:
        return None
    market_excess = pd.to_numeric(reg_data['market_excess'], errors='coerce')
    portfolio_excess = pd.to_numeric(reg_data['portfolio_excess'], errors='coerce')
    valid_mask = market_excess.notna() & portfolio_excess.notna()
    market_excess = market_excess[valid_mask].to_numpy(dtype=float)
    portfolio_excess = portfolio_excess[valid_mask].to_numpy(dtype=float)
    if len(market_excess) < 2:
        return None
    X = pd.DataFrame({'market_excess': market_excess})
    X = add_constant(X, has_constant='add')
    model = OLS(portfolio_excess, X).fit()
    n_months = len(portfolio_excess)
    arithmetic_monthly_return = pd.to_numeric(reg_data.loc[valid_mask, 'portfolio_return'], errors='coerce').mean()
    monthly_vol = portfolio_excess.std(ddof=1)
    rf_mean_sample = reg_data.loc[valid_mask, 'rf'].mean()
    params, tvalues, pvalues = model.params, model.tvalues, model.pvalues
    alpha = params.iloc[0]
    beta = params.iloc[1]
    alpha_tstat = tvalues.iloc[0]
    beta_tstat = tvalues.iloc[1]
    alpha_pval = pvalues.iloc[0]
    beta_pval = pvalues.iloc[1]
    ci = model.conf_int(alpha=0.05)
    return pd.Series({
        'arithmetic_monthly_return': arithmetic_monthly_return,
        'annualized_return': (1 + arithmetic_monthly_return) ** 12 - 1,
        'alpha': alpha, 'beta': beta,
        'alpha_tstat': alpha_tstat, 'beta_tstat': beta_tstat,
        'alpha_pval': alpha_pval, 'beta_pval': beta_pval,
        'alpha_lower_ci': ci.iloc[0, 0], 'alpha_upper_ci': ci.iloc[0, 1],
        'beta_lower_ci': ci.iloc[1, 0], 'beta_upper_ci': ci.iloc[1, 1],
        'r_squared': model.rsquared,
        'sharpe_ratio': (arithmetic_monthly_return - rf_mean_sample) / monthly_vol * np.sqrt(12) if monthly_vol > 0 else np.nan,
        'annualized_alpha': alpha * 12,
        'monthly_vol': monthly_vol,
        'annualized_vol': monthly_vol * np.sqrt(12),
        'n_observations': n_months
    })

def run_ff3_regression(portfolio_returns, market_returns, rf_rate):
    """Fama-French 3-factor: R - rf = alpha + b_mkt*mktrf + b_smb*smb + b_hml*hml."""
    for c in ['rf', 'mktrf', 'smb', 'hml']:
        if c not in market_returns.columns or market_returns[c].isna().all():
            raise ValueError(f"FF3 factor '{c}' must be in market_returns (from ff.factors_monthly).")
    portfolio_returns = portfolio_returns.copy()
    portfolio_returns['date'] = portfolio_returns['date'] + pd.offsets.MonthEnd(0)
    cols = ['date', 'mktrf', 'smb', 'hml', 'rf']
    market_data = market_returns[cols].copy()
    market_data['date'] = pd.to_datetime(market_data['date']) + pd.offsets.MonthEnd(0)
    reg_data = portfolio_returns.merge(market_data, on='date', how='left')
    reg_data = reg_data[reg_data['portfolio_return'].notna() & reg_data['mktrf'].notna() & reg_data['rf'].notna()].copy()
    reg_data['portfolio_excess'] = reg_data['portfolio_return'] - reg_data['rf'].astype(float)
    reg_data = reg_data[reg_data['portfolio_excess'].notna()].copy()
    if len(reg_data) < 5:
        return None
    y = reg_data['portfolio_excess'].astype(float).values
    X = reg_data[['mktrf', 'smb', 'hml']].astype(float)
    X = add_constant(X, has_constant='add')
    model = OLS(y, X).fit()
    n_months = len(y)
    arithmetic_monthly_return = reg_data['portfolio_return'].mean()
    monthly_vol = reg_data['portfolio_excess'].std(ddof=1)
    rf_mean = reg_data['rf'].mean()
    params, tvalues, pvalues, ci = model.params, model.tvalues, model.pvalues, model.conf_int(alpha=0.05)
    return pd.Series({
        'arithmetic_monthly_return': arithmetic_monthly_return,
        'annualized_return': (1 + arithmetic_monthly_return) ** 12 - 1,
        'alpha': params.iloc[0], 'beta': params.iloc[1],
        'beta_mkt': params.iloc[1], 'beta_smb': params.iloc[2], 'beta_hml': params.iloc[3],
        'alpha_tstat': tvalues.iloc[0], 'beta_tstat': tvalues.iloc[1],
        'beta_smb_tstat': tvalues.iloc[2], 'beta_hml_tstat': tvalues.iloc[3],
        'alpha_pval': pvalues.iloc[0], 'beta_pval': pvalues.iloc[1],
        'alpha_lower_ci': ci.iloc[0, 0], 'alpha_upper_ci': ci.iloc[0, 1],
        'beta_lower_ci': ci.iloc[1, 0], 'beta_upper_ci': ci.iloc[1, 1],
        'r_squared': model.rsquared,
        'sharpe_ratio': (arithmetic_monthly_return - rf_mean) / monthly_vol * np.sqrt(12) if monthly_vol > 0 else np.nan,
        'monthly_vol': monthly_vol,
        'annualized_vol': monthly_vol * np.sqrt(12),
        'n_observations': n_months
    })

def calculate_sharpe_ratio(returns, rf_rate):
    if len(returns) == 0 or returns.isna().all():
        return np.nan
    excess = returns - rf_rate
    sd = returns.std()
    if sd == 0 or pd.isna(sd):
        return np.nan
    return (excess.mean() / sd) * np.sqrt(12)

def calculate_all_portfolio_metrics(portfolio_data, market_data, portfolio_name, rf_rate, model='capm'):
    """model: 'capm' or 'ff3'."""
    group_col = portfolio_data.columns[1]
    run_reg = (lambda pr, md, rf: run_capm_regression(pr, md, rf, use_ew_benchmark=False)) if model == 'capm' else run_ff3_regression
    
    def is_long_short_portfolio(group_name):
        """Detect if this is a long-short spread portfolio (Q5-Q1 or Q5-No R&D)."""
        group_str = str(group_name)
        return group_str in ['Q5-Q1', 'Q5-No R&D']
    
    ew_returns = portfolio_data[['date', 'ew_return', group_col]].copy()
    ew_returns = ew_returns.rename(columns={'ew_return': 'portfolio_return'})
    ew_metrics_list = []
    for group_name, group_data in ew_returns.groupby(group_col):
        # For long-short portfolios, set rf=0 to avoid double-subtracting
        reg_market_data = market_data.copy()
        if is_long_short_portfolio(group_name):
            reg_market_data['rf'] = 0.0
        r = run_reg(group_data[['date', 'portfolio_return']], reg_market_data, rf_rate)
        if r is not None and len(r) > 0:
            ew_metrics_list.append({**r.to_dict(), group_col: group_name})
    ew_metrics = pd.DataFrame(ew_metrics_list)
    if len(ew_metrics) > 0:
        ew_metrics['weighting'] = 'Equal-Weighted'
    vw_returns = portfolio_data[['date', 'vw_return', group_col]].copy()
    vw_returns = vw_returns.rename(columns={'vw_return': 'portfolio_return'})
    vw_metrics_list = []
    for group_name, group_data in vw_returns.groupby(group_col):
        # For long-short portfolios, set rf=0 to avoid double-subtracting
        reg_market_data = market_data.copy()
        if is_long_short_portfolio(group_name):
            reg_market_data['rf'] = 0.0
        r = run_reg(group_data[['date', 'portfolio_return']], reg_market_data, rf_rate)
        if r is not None and len(r) > 0:
            vw_metrics_list.append({**r.to_dict(), group_col: group_name})
    vw_metrics = pd.DataFrame(vw_metrics_list)
    if len(vw_metrics) > 0:
        vw_metrics['weighting'] = 'Value-Weighted'
    if len(ew_metrics) > 0 and len(vw_metrics) > 0:
        all_metrics = pd.concat([ew_metrics, vw_metrics], ignore_index=True)
    elif len(ew_metrics) > 0:
        all_metrics = ew_metrics
    elif len(vw_metrics) > 0:
        all_metrics = vw_metrics
    else:
        return pd.DataFrame()
    all_metrics['portfolio'] = all_metrics['weighting'] + '_' + all_metrics[group_col].astype(str)
    all_metrics = all_metrics.drop(columns=[group_col])
    return all_metrics

def format_performance_table(metrics, model='capm'):
    if metrics is None or (isinstance(metrics, pd.DataFrame) and (metrics.empty or 'portfolio' not in metrics.columns)):
        return metrics if isinstance(metrics, pd.DataFrame) else pd.DataFrame()
    rounding_map = {
        'arithmetic_monthly_return': 6, 'annualized_return': 4, 'alpha': 6, 'beta': 6,
        'alpha_tstat': 6, 'beta_tstat': 2, 'alpha_lower_ci': 6, 'alpha_upper_ci': 6,
        'beta_lower_ci': 6, 'beta_upper_ci': 6, 'r_squared': 6, 'sharpe_ratio': 6,
        'monthly_vol': 6, 'annualized_vol': 6,
        'beta_mkt': 6, 'beta_smb': 6, 'beta_hml': 6, 'beta_smb_tstat': 2, 'beta_hml_tstat': 2
    }
    table = metrics.copy()
    if 'portfolio' not in table.columns and 'weighting' in table.columns:
        group_col = [c for c in ['portfolio_id', 'research_not'] if c in table.columns]
        if group_col:
            table['portfolio'] = table['weighting'] + '_' + table[group_col[0]].astype(str)
    table['Portfolio'] = table['portfolio']
    table = format_numeric_cols(table, rounding_map)
    table['Alpha p-value'] = table['alpha_pval'].apply(lambda x: f"{x:.3e}")
    table['Beta p-value'] = table['beta_pval'].apply(lambda x: f"{x:.3e}")
    base_cols = ['Portfolio', 'arithmetic_monthly_return', 'monthly_vol', 'annualized_return', 'alpha', 'beta', 'alpha_tstat', 'beta_tstat', 'Alpha p-value', 'Beta p-value', 'alpha_lower_ci', 'alpha_upper_ci', 'beta_lower_ci', 'beta_upper_ci', 'r_squared', 'sharpe_ratio']
    renames = {'arithmetic_monthly_return': 'Mean Return (monthly)', 'monthly_vol': 'Std Return (monthly)', 'annualized_return': 'Annualized Return', 'alpha': 'Alpha (Monthly)', 'beta': 'Beta (Mkt)', 'alpha_tstat': 'Alpha t-stat', 'beta_tstat': 'Beta t-stat', 'alpha_lower_ci': 'Alpha Lower CI (95%)', 'alpha_upper_ci': 'Alpha Upper CI (95%)', 'beta_lower_ci': 'Beta Lower CI (95%)', 'beta_upper_ci': 'Beta Upper CI (95%)', 'r_squared': 'R-squared', 'sharpe_ratio': 'Sharpe Ratio'}
    if model == 'ff3' and 'beta_smb' in table.columns:
        base_cols = base_cols[:5] + ['beta_smb', 'beta_hml', 'beta_smb_tstat', 'beta_hml_tstat'] + base_cols[5:]
        renames['beta_smb'], renames['beta_hml'] = 'SMB Beta', 'HML Beta'
        renames['beta_smb_tstat'], renames['beta_hml_tstat'] = 'SMB t-stat', 'HML t-stat'
    out = table[[c for c in base_cols if c in table.columns]].copy()
    return out.rename(columns=renames)

def create_alpha_table(portfolio_metrics, market_returns, rf_rate_monthly, weighting=None):
    """Build alpha table. If weighting is 'Equal-Weighted' or 'Value-Weighted', one table per scheme with columns: Portfolio, Mean Return (%), Std Dev (%), Alpha (monthly %), t-statistic, Beta, R², Sharpe. Row order: No R&D, Q1..Q5, Q5-Q1, Q5-No R&D."""
    benchmark_sharpe_vw = calculate_sharpe_ratio(market_returns['vwretd'], rf_rate_monthly)
    def portfolio_display_name(x):
        parts = x.split('_', 1)
        if len(parts) != 2:
            return x, 'Unknown'
        wname = 'Value-Weighted' if 'Value' in parts[0] else 'Equal-Weighted'
        return parts[1], wname
    alpha_table = portfolio_metrics.copy()
    alpha_table[['Portfolio', 'Weighting']] = alpha_table['portfolio'].apply(lambda x: pd.Series(portfolio_display_name(x)))
    if weighting is not None:
        alpha_table = alpha_table[alpha_table['Weighting'] == weighting].copy()
        if alpha_table.empty:
            return alpha_table
    alpha_table['alpha_monthly_pct'] = (alpha_table['alpha'] * 100).round(2)
    alpha_table['alpha_star'] = alpha_table['alpha_pval'].apply(lambda x: '***' if x < 0.01 else '**' if x < 0.05 else '*' if x < 0.10 else '')
    alpha_table['Alpha (monthly %)'] = alpha_table['alpha_monthly_pct'].apply(lambda x: f"{x:.2f}") + alpha_table['alpha_star']
    alpha_table['Mean Return (%)'] = (alpha_table['arithmetic_monthly_return'] * 100).round(2)
    alpha_table['Std Dev (%)'] = (alpha_table['monthly_vol'] * 100).round(2)
    alpha_table['t-statistic'] = alpha_table['alpha_tstat'].round(2)
    alpha_table['Beta'] = alpha_table['beta'].round(2)
    is_ff3 = 'beta_smb' in alpha_table.columns and 'beta_hml' in alpha_table.columns
    if is_ff3:
        alpha_table['Beta (Mkt)'] = alpha_table['beta'].round(2)
        alpha_table['Beta (SMB)'] = alpha_table['beta_smb'].round(2)
        alpha_table['Beta (HML)'] = alpha_table['beta_hml'].round(2)
    alpha_table['R²'] = alpha_table['r_squared'].round(2)
    alpha_table['Sharpe'] = alpha_table['sharpe_ratio'].round(2)
    alpha_table['is_significant'] = alpha_table['alpha_pval'] < 0.10
    alpha_table['sort_key'] = alpha_table['Weighting'] + '_' + alpha_table['Portfolio']
    sort_map = {'Equal-Weighted_No R&D': 0, 'Equal-Weighted_Q1': 1, 'Equal-Weighted_Q2': 2, 'Equal-Weighted_Q3': 3,
                'Equal-Weighted_Q4': 4, 'Equal-Weighted_Q5': 5, 'Equal-Weighted_Q5-Q1': 6, 'Equal-Weighted_Q5-No R&D': 7,
                'Value-Weighted_No R&D': 8, 'Value-Weighted_Q1': 9, 'Value-Weighted_Q2': 10, 'Value-Weighted_Q3': 11,
                'Value-Weighted_Q4': 12, 'Value-Weighted_Q5': 13, 'Value-Weighted_Q5-Q1': 14, 'Value-Weighted_Q5-No R&D': 15}
    alpha_table['sort_order'] = alpha_table['sort_key'].map(sort_map).fillna(99)
    alpha_table = alpha_table.sort_values('sort_order')
    if weighting is not None:
        display_cols = ['Portfolio', 'Mean Return (%)', 'Std Dev (%)', 'Alpha (monthly %)', 't-statistic']
        if is_ff3:
            display_cols += ['Beta (Mkt)', 'Beta (SMB)', 'Beta (HML)']
        else:
            display_cols += ['Beta']
        display_cols += ['R²', 'Sharpe', 'is_significant', 'sort_order']
        benchmark_cols = ['Portfolio', 'Mean Return (%)', 'Std Dev (%)', 'Alpha (monthly %)', 't-statistic']
        if is_ff3:
            benchmark_cols += ['Beta (Mkt)', 'Beta (SMB)', 'Beta (HML)']
        else:
            benchmark_cols += ['Beta']
        benchmark_cols += ['R²', 'Sharpe', 'is_significant', 'sort_order']
    else:
        display_cols = ['Weighting', 'Portfolio', 'Mean Return (%)', 'Std Dev (%)', 'Alpha (monthly %)', 't-statistic', 'p-value']
        if is_ff3:
            display_cols += ['Beta (Mkt)', 'Beta (SMB)', 'Beta (HML)']
        else:
            display_cols += ['Beta']
        display_cols += ['R²', 'Sharpe', 'is_significant', 'sort_order']
        benchmark_cols = None
    alpha_table_filtered = alpha_table[[c for c in display_cols if c in alpha_table.columns]].copy()
    alpha_table_str = alpha_table_filtered.drop(columns='sort_order').astype(str)
    alpha_table_str['sort_order'] = alpha_table_filtered['sort_order']
    if benchmark_cols is not None:
        bench = {c: ['—'] for c in display_cols if c not in ('Sharpe', 'sort_order', 'is_significant')}
        bench['Portfolio'] = ['CRSP VW Benchmark']
        bench['Sharpe'] = [round(benchmark_sharpe_vw, 2)]
        bench['is_significant'] = [False]
        bench['sort_order'] = [99]
        benchmark_rows = pd.DataFrame(bench)
    else:
        if is_ff3:
            benchmark_rows = pd.DataFrame({
                'Weighting': ['—'], 'Portfolio': ['CRSP VW Benchmark'], 'Mean Return (%)': ['—'], 'Std Dev (%)': ['—'],
                'Alpha (monthly %)': ['—'], 't-statistic': ['—'], 'p-value': ['—'],
                'Beta (Mkt)': ['—'], 'Beta (SMB)': ['—'], 'Beta (HML)': ['—'], 'R²': ['—'],
                'Sharpe': [round(benchmark_sharpe_vw, 2)], 'is_significant': [False], 'sort_order': [99]
            })
        else:
            benchmark_rows = pd.DataFrame({
                'Weighting': ['—'], 'Portfolio': ['CRSP VW Benchmark'], 'Mean Return (%)': ['—'], 'Std Dev (%)': ['—'],
                'Alpha (monthly %)': ['—'], 't-statistic': ['—'], 'p-value': ['—'],
                'Beta': ['—'], 'R²': ['—'],
                'Sharpe': [round(benchmark_sharpe_vw, 2)], 'is_significant': [False], 'sort_order': [99]
            })
    benchmark_rows['sort_order'] = benchmark_rows['sort_order'].astype(float)
    alpha_table_str = pd.concat([alpha_table_str, benchmark_rows], ignore_index=True)
    alpha_table_str = alpha_table_str.sort_values('sort_order').drop(columns='sort_order')
    return alpha_table_str

def create_alpha_table_html(alpha_table, custom_css, caption_suffix='', table_type=""):
    """Create HTML alpha table with same styling (color-coded rows). Columns taken from alpha_table."""
    alpha_table_display = alpha_table.drop(columns='is_significant', errors='ignore')
    display_cols = [c for c in alpha_table_display.columns]
    if table_type == 'FF3':
        caption_title = "Alpha Estimation Results: Three-Factor Model"
        caption_eq = "Rp - Rƒ = α + β_mkt(MKT-Rƒ) + β_smb SMB + β_hml HML"
    else:
        caption_title = "Alpha Estimation Results: Single-Factor Model"
        caption_eq = "Rp - Rƒ = α + β(R_CRSP_VW - Rƒ)"
    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
    {custom_css}
    </head>
    <body>
    <table class="table table-dark table-striped">
    <caption><b>{caption_title}</b><br>
    <i>{caption_eq}</i>{caption_suffix}<br>
    <i>{table_type}</i></caption>
    <thead>
    <tr>
    """ + "".join(f"<th>{c}</th>" for c in display_cols) + """
    </tr>
    </thead>
    <tbody>
    """
    for _, row in alpha_table_display.iterrows():
        portfolio = row['Portfolio']
        if portfolio == 'CRSP VW Benchmark':
            color = 'white'
        elif 'Q5-No R&D' in portfolio: color = '#00b4d8'
        elif 'Q5-Q1' in portfolio: color = '#00c9a7'
        elif 'No R&D' in portfolio:
            color = '#5dade2'
        elif 'Q1' in portfolio: color = '#ffa600'
        elif 'Q2' in portfolio: color = '#dd5182'
        elif 'Q3' in portfolio: color = '#955196'
        elif 'Q4' in portfolio: color = '#bc5090'
        elif 'Q5' in portfolio: color = '#ff6361'
        else:
            color = 'white'
        bold = 'font-weight: bold;' if portfolio == 'CRSP VW Benchmark' or 'No R&D' in portfolio or 'Q5-Q1' in portfolio or any(f'Q{i}' in portfolio for i in range(1,6)) else ''
        cells = "".join(f"<td>{row[c]}</td>" for c in display_cols)
        html += f"""
        <tr style="color: {color}; {bold}">
        {cells}
        </tr>
        """
    html += """
    </tbody>
    </table>
    </body>
    </html>
    """
    return html

custom_css = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Tomorrow:wght@400;600;700&display=swap');
body { font-family: 'Tomorrow', sans-serif; background-color: #1d1d1c; padding: 1rem; margin: 0; color: white; }
.table-dark.table-striped tbody tr:nth-of-type(odd) { background-color: rgba(255, 255, 255, 0.05) !important; }
.table-dark.table-striped tbody tr:nth-of-type(even) { background-color: rgba(0, 0, 0, 0.15) !important; }
.table-dark tbody tr[style*='color'] { background-color: inherit !important; }
.table-dark td, .table-dark th { padding: 0.5rem 0.75rem !important; }
</style>
"""

def create_performance_chart(chart_data, panel_type='vw', colors=None):
    pattern = 'vw_' if panel_type == 'vw' else 'ew_'
    benchmark_name = 'CRSP VW Benchmark'
    panel_data = chart_data[chart_data['key'].str.contains(pattern, na=False)].copy()
    def key_to_name(x):
        if 'CRSP' in str(x): return benchmark_name
        if '_' in str(x): return str(x).split('_', 1)[-1]
        return x
    panel_data['portfolio_name'] = panel_data['key'].apply(key_to_name)
    # Enforce portfolio display order
    portfolio_order = ['No R&D', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q5-Q1', 'Q5-No R&D', 'CRSP VW Benchmark']
    ordered_portfolios = [p for p in portfolio_order if p in panel_data['portfolio_name'].unique()]
    # Add any remaining portfolios not in the explicit order
    ordered_portfolios += [p for p in panel_data['portfolio_name'].unique() if p not in ordered_portfolios]
    fig = go.Figure()
    for portfolio in ordered_portfolios:
        port_data = panel_data[panel_data['portfolio_name'] == portfolio].sort_values('date')
        color = (colors or {}).get(portfolio, '#888')
        fig.add_trace(go.Scatter(x=port_data['date'], y=port_data['cumvalue'], mode='lines', name=portfolio, line=dict(color=color, width=3)))
    fig.update_layout(title='Cumulative Performance (1980-2022)', xaxis_title='Date', yaxis_title='Cumulative Value ($)',
                      yaxis=dict(rangemode='tozero', tickformat='$.2f'), template='plotly_dark', height=500)
    return fig

def create_combined_performance_chart(chart_data, colors=None):
    """Create a single chart with EW portfolios first, then VW portfolios, matching the requested ordering."""
    benchmark_name = 'CRSP VW Benchmark'
    portfolio_order = ['No R&D', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q5-Q1', 'Q5-No R&D']
    def key_to_name(x):
        if 'CRSP' in str(x): return benchmark_name
        if '_' in str(x): return str(x).split('_', 1)[-1]
        return x
    fig = go.Figure()
    # EW portfolios first
    ew_data = chart_data[chart_data['key'].str.contains('ew_', na=False)].copy()
    ew_data['portfolio_name'] = ew_data['key'].apply(key_to_name)
    ew_ordered = [p for p in portfolio_order if p in ew_data['portfolio_name'].unique()]
    for portfolio in ew_ordered:
        port_data = ew_data[ew_data['portfolio_name'] == portfolio].sort_values('date')
        color = (colors or {}).get(portfolio, '#888')
        fig.add_trace(go.Scatter(x=port_data['date'], y=port_data['cumvalue'], mode='lines', name=f'{portfolio} EW', line=dict(color=color, width=2, dash='solid')))
    # VW portfolios second
    vw_data = chart_data[chart_data['key'].str.contains('vw_', na=False)].copy()
    vw_data['portfolio_name'] = vw_data['key'].apply(key_to_name)
    vw_ordered = [p for p in portfolio_order if p in vw_data['portfolio_name'].unique()]
    for portfolio in vw_ordered:
        port_data = vw_data[vw_data['portfolio_name'] == portfolio].sort_values('date')
        color = (colors or {}).get(portfolio, '#888')
        fig.add_trace(go.Scatter(x=port_data['date'], y=port_data['cumvalue'], mode='lines', name=f'{portfolio} VW', line=dict(color=color, width=2, dash='dash')))
    # Benchmark last
    bm_data = chart_data[chart_data['key'].str.contains('CRSP', na=False)].copy()
    if not bm_data.empty:
        bm_data = bm_data.sort_values('date')
        color = (colors or {}).get(benchmark_name, '#58508d')
        fig.add_trace(go.Scatter(x=bm_data['date'], y=bm_data['cumvalue'], mode='lines', name=benchmark_name, line=dict(color=color, width=3, dash='dot')))
    fig.update_layout(title='Cumulative Performance (1980-2022)', xaxis_title='Date', yaxis_title='Cumulative Value ($)',
                      yaxis=dict(rangemode='tozero', tickformat='$.2f'), template='plotly_dark', height=600)
    return fig

def create_longshort_performance_chart(chart_data, colors=None):
    """Create a chart showing only the long-short portfolios (Q5-Q1 and Q5-No R&D)."""
    def key_to_name(x):
        if '_' in str(x): return str(x).split('_', 1)[-1]
        return x
    
    # Filter for only long-short portfolios
    longshort_portfolios = ['Q5-Q1', 'Q5-No R&D']
    
    fig = go.Figure()
    
    # EW long-short portfolios
    ew_data = chart_data[chart_data['key'].str.contains('ew_', na=False)].copy()
    ew_data['portfolio_name'] = ew_data['key'].apply(key_to_name)
    for portfolio in longshort_portfolios:
        if portfolio in ew_data['portfolio_name'].unique():
            port_data = ew_data[ew_data['portfolio_name'] == portfolio].sort_values('date')
            color = (colors or {}).get(portfolio, '#888')
            fig.add_trace(go.Scatter(x=port_data['date'], y=port_data['cumvalue'], mode='lines', 
                                    name=f'{portfolio} (EW)', line=dict(color=color, width=3, dash='solid')))
    
    # VW long-short portfolios
    vw_data = chart_data[chart_data['key'].str.contains('vw_', na=False)].copy()
    vw_data['portfolio_name'] = vw_data['key'].apply(key_to_name)
    for portfolio in longshort_portfolios:
        if portfolio in vw_data['portfolio_name'].unique():
            port_data = vw_data[vw_data['portfolio_name'] == portfolio].sort_values('date')
            color = (colors or {}).get(portfolio, '#888')
            fig.add_trace(go.Scatter(x=port_data['date'], y=port_data['cumvalue'], mode='lines', 
                                    name=f'{portfolio} (VW)', line=dict(color=color, width=3, dash='dash')))
    
    fig.update_layout(title='Long-Short Portfolios: Cumulative Performance (1980-2022)', 
                      xaxis_title='Date', yaxis_title='Cumulative Value ($)',
                      yaxis=dict(rangemode='tozero', tickformat='$.2f'), 
                      template='plotly_dark', height=500)
    return fig


## Build Market Returns and Portfolio Metrics

In [14]:
mkt['date'] = pd.to_datetime(mkt['date']).dt.to_period('M').dt.to_timestamp()
ff['date'] = pd.to_datetime(ff['date']).dt.to_period('M').dt.to_timestamp()
for c in ['rf', 'mktrf', 'smb', 'hml']:
    ff[c] = pd.to_numeric(ff[c], errors='coerce')
market_returns = mkt.merge(ff, on='date', how='left')
if 'rf' not in market_returns.columns or market_returns['rf'].isna().all():
    raise ValueError("Risk-free rate 'rf' must be present (from ff.factors_monthly).")
market_returns['vwretd'] = pd.to_numeric(market_returns['vwretd'], errors='coerce')
market_returns['ewretd'] = pd.to_numeric(market_returns['ewretd'], errors='coerce')

rf_rate_monthly = market_returns['rf'].mean()

def run_quintile_metrics(analysis_data, market_returns, rf_rate_monthly):
    """Portfolio returns (with Q5-Q1 and Q5-No R&D spreads) and CAPM/FF3 metrics for one analysis_data."""
    pr = calculate_portfolio_returns(analysis_data, 'portfolio_id')
    pr = add_long_short_spreads(pr, 'portfolio_id')
    pm_capm = calculate_all_portfolio_metrics(pr, market_returns, 'R&D Quintiles', rf_rate_monthly, model='capm')
    pm_ff3 = calculate_all_portfolio_metrics(pr, market_returns, 'R&D Quintiles', rf_rate_monthly, model='ff3')
    return pr, pm_capm, pm_ff3

def metrics_for_period(portfolio_returns, market_returns, rf_rate_monthly, start_date, end_date):
    """Compute CAPM and FF3 metrics for a subperiod (start_date, end_date inclusive)."""
    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)
    pr = portfolio_returns[(portfolio_returns['date'] >= start_ts) & (portfolio_returns['date'] <= end_ts)].copy()
    mr = market_returns[(market_returns['date'] >= start_ts) & (market_returns['date'] <= end_ts)].copy()
    if pr.empty or mr.empty:
        return None, None
    pm_capm = calculate_all_portfolio_metrics(pr, mr, 'R&D Quintiles', rf_rate_monthly, model='capm')
    pm_ff3 = calculate_all_portfolio_metrics(pr, mr, 'R&D Quintiles', rf_rate_monthly, model='ff3')
    return pm_capm, pm_ff3

SUBPERIODS = [
    (f'{START_DATE[:4]}-{SPLIT_DATE[:4]}', START_DATE, SPLIT_DATE),
    (f'{int(SPLIT_DATE[:4])+1}-{END_DATE[:4]}', f'{int(SPLIT_DATE[:4])+1}-01-01', END_DATE),
]

portfolio_returns_norm, portfolio_metrics_capm_norm, portfolio_metrics_ff3_norm = run_quintile_metrics(analysis_data_norm, market_returns, rf_rate_monthly)

/var/folders/fj/plbn4w9d6fs55651zh_pxsd80000gn/T/ipykernel_23034/2524599387.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  portfolio_returns = data.groupby(['date', group_var]).apply(


In [15]:
# Contribution to return by sector (GICS) for each portfolio — period-by-period, for CSV extract and area chart
def compute_contribution_to_return(analysis_data):
    """Long format: date, portfolio_id, weighting, sector, contribution (period return contribution, not cumulative)."""
    if 'sector' not in analysis_data.columns:
        analysis_data = analysis_data.copy()
        analysis_data['sector'] = 'Unknown'
    ad = analysis_data.dropna(subset=['ret', 'mktcap_lag', 'portfolio_id']).copy()
    ad['date'] = pd.to_datetime(ad['date']) + pd.offsets.MonthEnd(0)
    rows = []
    for (date, pid), g in ad.groupby(['date', 'portfolio_id']):
        g = g[g['mktcap_lag'] > 0]
        if g.empty:
            continue
        total_mkt = g['mktcap_lag'].sum()
        n = len(g)
        g = g.copy()
        g['vw_w'] = g['mktcap_lag'] / total_mkt
        g['ew_w'] = 1.0 / n
        g['vw_contrib'] = g['vw_w'] * g['ret']
        g['ew_contrib'] = g['ew_w'] * g['ret']
        for sector, sg in g.groupby('sector', dropna=False):
            sector = sector if pd.notna(sector) else 'Unknown'
            rows.append({'date': date, 'portfolio_id': pid, 'weighting': 'Value-Weighted', 'sector': sector, 'contribution': sg['vw_contrib'].sum()})
            rows.append({'date': date, 'portfolio_id': pid, 'weighting': 'Equal-Weighted', 'sector': sector, 'contribution': sg['ew_contrib'].sum()})
    out = pd.DataFrame(rows)
    out['date'] = pd.to_datetime(out['date']).dt.strftime('%Y-%m-%d')
    return out

contributions_df = compute_contribution_to_return(analysis_data_norm)
OUTPUT_CSV = Path('notes/portfolio_contributions_to_return.csv')
contributions_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(contributions_df)} rows to {OUTPUT_CSV}")

# Annual contribution: sum by year, portfolio_id, weighting, sector
contributions_df['year'] = pd.to_datetime(contributions_df['date']).dt.year
contributions_annual = (
    contributions_df.groupby(['year', 'portfolio_id', 'weighting', 'sector'], as_index=False)['contribution'].sum()
)
contributions_annual['year'] = contributions_annual['year'].astype(int)

# Write HTML with annual data embedded to avoid CORS when opening file://
data_json = contributions_annual.to_json(orient='records')
html_content = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Annual contribution to return by sector</title>
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
  <style>
    body { font-family: system-ui, sans-serif; margin: 1rem; background: #1a1a1a; color: #e0e0e0; }
    h1 { font-size: 1.25rem; }
    .controls { display: flex; gap: 1rem; align-items: center; margin-bottom: 1rem; flex-wrap: wrap; }
    label { display: flex; align-items: center; gap: 0.5rem; }
    select { padding: 0.35rem 0.75rem; background: #2d2d2d; color: #e0e0e0; border: 1px solid #444; border-radius: 4px; }
    #chart { width: 100%; height: 80vh; }
    .note { font-size: 0.85rem; color: #888; margin-top: 0.5rem; }
  </style>
</head>
<body>
  <h1>Annual contribution to return by sector</h1>
  <div class="controls">
    <label>Portfolio: <select id="portfolio"></select></label>
    <label>Weighting: <select id="weighting">
      <option value="Value-Weighted">Value-Weighted</option>
      <option value="Equal-Weighted">Equal-Weighted</option>
    </select></label>
  </div>
  <div id="chart"></div>
  <p class="note">Stacked bar = sector contribution to portfolio return per year (sum of monthly contributions). Data embedded; re-run notebook to refresh.</p>
  <script>
    window.CONTRIBUTION_DATA = ''' + data_json + ''';
  </script>
  <script>
    (function () {
      const portfolioSelect = document.getElementById('portfolio');
      const weightingSelect = document.getElementById('weighting');
      const chartDiv = document.getElementById('chart');
      const rawData = window.CONTRIBUTION_DATA || [];
      const portfolios = [...new Set(rawData.map(r => r.portfolio_id))].filter(Boolean).sort();
      portfolioSelect.innerHTML = portfolios.map(p => '<option value="' + p + '">' + p + '</option>').join('');
      function updateChart() {
        const portfolio = portfolioSelect.value;
        const weighting = weightingSelect.value;
        const filtered = rawData.filter(r => r.portfolio_id === portfolio && r.weighting === weighting);
        if (filtered.length === 0) { chartDiv.innerHTML = '<p>No data for this selection.</p>'; return; }
        const byYear = {};
        filtered.forEach(r => {
          const y = r.year;
          if (!byYear[y]) byYear[y] = {};
          byYear[y][r.sector] = Number(r.contribution) || 0;
        });
        const years = Object.keys(byYear).map(Number).sort((a,b) => a - b);
        const sectors = [...new Set(filtered.map(r => r.sector))].sort();
        const traces = sectors.map(sector => ({
          name: sector,
          x: years,
          y: years.map(y => byYear[y][sector] ?? 0),
          type: 'bar',
          stack: 'one'
        }));
        const layout = {
          title: { text: 'Annual contribution to return by sector — ' + portfolio + ' (' + weighting + ')', font: { size: 14 } },
          margin: { t: 40, r: 40, b: 40, l: 50 },
          xaxis: { title: 'Year', type: 'category' },
          yaxis: { title: 'Contribution (annual return)', tickformat: '.2%' },
          barmode: 'stack',
          showlegend: true,
          legend: { orientation: 'h', y: 1.08 },
          paper_bgcolor: '#1a1a1a',
          plot_bgcolor: '#252525',
          font: { color: '#e0e0e0' },
          hovermode: 'x unified'
        };
        Plotly.newPlot(chartDiv, traces, layout, { responsive: true });
      }
      portfolioSelect.addEventListener('change', updateChart);
      weightingSelect.addEventListener('change', updateChart);
      updateChart();
    })();
  </script>
</body>
</html>'''
Path('notes/portfolio_contributions_viz.html').write_text(html_content, encoding='utf-8')
print("Wrote notes/portfolio_contributions_viz.html (annual contribution chart; open directly in browser, no CORS).")

Saved 62632 rows to notes/portfolio_contributions_to_return.csv
Wrote notes/portfolio_contributions_viz.html (annual contribution chart; open directly in browser, no CORS).


## Outputs: Performance Tables, Alpha Table, Chart

In [ ]:
from IPython.display import HTML, display

TEST_PERIOD = f'{START_DATE[:4]}–{END_DATE[:4]}'
colors = {'CRSP VW Benchmark': '#58508d', 'No R&D': '#5dade2', 'Q1': '#ffa600', 'Q2': '#dd5182', 'Q3': '#955196', 'Q4': '#bc5090', 'Q5': '#ff6361', 'Q5-No R&D': '#00b4d8', 'Q5-Q1': '#00c9a7'}
market_vw = market_returns.assign(date=market_returns['date'] + pd.offsets.MonthEnd(0), key='vw_CRSP_VW_Benchmark', value=market_returns['vwretd'])
market_ew = market_returns.assign(date=market_returns['date'] + pd.offsets.MonthEnd(0), key='ew_CRSP_VW_Benchmark', value=market_returns['vwretd'])

def _show(period_label, pm_capm, pm_ff3, rd_label):
    for model_name, pm in [('CAPM', pm_capm), ('FF3', pm_ff3)]:
        for weighting in ['Equal-Weighted', 'Value-Weighted']:
            alpha_table = create_alpha_table(pm, market_returns, rf_rate_monthly, weighting=weighting)
            if alpha_table.empty:
                continue
            caption_suffix = f'<br>{rd_label}<br>Sample Period: {period_label} | Weighting Scheme: {weighting}'
            display(HTML(create_alpha_table_html(alpha_table, custom_css, caption_suffix=caption_suffix, table_type=model_name)))

variants = [
    ('Normalized (XRDW % of ME)', portfolio_returns_norm, portfolio_metrics_capm_norm, portfolio_metrics_ff3_norm),
]
returns_data_norm = None

for rd_label, pr, pm_capm, pm_ff3 in variants:
    _show(TEST_PERIOD, pm_capm, pm_ff3, rd_label)
    for period_label, start_date, end_date in SUBPERIODS:
        pm_capm_sub, pm_ff3_sub = metrics_for_period(pr, market_returns, rf_rate_monthly, start_date, end_date)
        if pm_capm_sub is None or pm_capm_sub.empty:
            continue
        _show(period_label, pm_capm_sub, pm_ff3_sub, rd_label)

    chart_data = (pr.assign(key=pr['portfolio_id'].astype(str))
        .melt(id_vars=['date', 'portfolio_id', 'key'], value_vars=['ew_return', 'vw_return'], var_name='weighting', value_name='value')
        .assign(key=lambda x: x['weighting'].str.replace('_return', '') + '_' + x['portfolio_id'].astype(str)))
    chart_data = pd.concat([chart_data[['date', 'key', 'value']], market_vw[['date', 'key', 'value']], market_ew[['date', 'key', 'value']]], ignore_index=True)
    t0 = chart_data.groupby('key')['date'].min().max()
    chart_data = chart_data[chart_data['date'] >= t0].sort_values(['key', 'date'])
    chart_data['cumvalue'] = (1 + pd.to_numeric(chart_data['value'], errors='coerce')).groupby(chart_data['key']).cumprod()
    chart_data['cumvalue'] = chart_data.groupby('key')['cumvalue'].transform(lambda x: x / x.iloc[0])
    # EW chart first
    fig_ew = create_performance_chart(chart_data, panel_type='ew', colors=colors)
    fig_ew.update_layout(title=f'Cumulative Performance ({START_DATE[:4]}-{END_DATE[:4]}) — Equal-Weighted — {rd_label}')
    fig_ew.show()
    # VW chart second
    fig_vw = create_performance_chart(chart_data, panel_type='vw', colors=colors)
    fig_vw.update_layout(title=f'Cumulative Performance ({START_DATE[:4]}-{END_DATE[:4]}) — Value-Weighted — {rd_label}')
    fig_vw.show()
    # Long-Short portfolios chart
    fig_ls = create_longshort_performance_chart(chart_data, colors=colors)
    fig_ls.update_layout(title=f'Long-Short Portfolios: Cumulative Performance ({START_DATE[:4]}-{END_DATE[:4]}) — {rd_label}')
    fig_ls.show()

    ret_wide = pr.pivot(index='date', columns='portfolio_id', values='ew_return').add_suffix('_ew').join(
        pr.pivot(index='date', columns='portfolio_id', values='vw_return').add_suffix('_vw')).reset_index()
    returns_data_norm = ret_wide

Portfolio,Mean Return (%),Std Dev (%),Alpha (monthly %),t-statistic,Beta,R²,Sharpe
No R&D,1.08,5.81,0.04,0.31,1.09,0.72,0.45
Q1,0.6,6.21,-0.52***,-3.92,1.21,0.77,0.15
Q2,0.99,6.38,-0.14,-0.97,1.22,0.75,0.36
Q3,1.03,6.68,-0.11,-0.7,1.25,0.71,0.37
Q4,1.39,7.45,0.22,1.07,1.3,0.63,0.5
Q5,2.34,9.81,1.07***,3.3,1.44,0.44,0.71
Q5-Q1,1.74,5.98,1.51***,5.71,0.24,0.03,1.01
Q5-No R&D,1.26,5.97,0.91***,3.54,0.35,0.07,0.73
CRSP VW Benchmark,—,—,—,—,—,—,0.42


Portfolio,Mean Return (%),Std Dev (%),Alpha (monthly %),t-statistic,Beta,R²,Sharpe
No R&D,0.94,4.37,0.02,0.27,0.92,0.91,0.49
Q1,0.83,5.17,-0.17*,-1.71,1.03,0.81,0.34
Q2,1.2,5.38,0.17*,1.65,1.08,0.82,0.56
Q3,1.17,5.54,0.12,1.13,1.11,0.82,0.53
Q4,1.29,6.22,0.19,1.37,1.18,0.74,0.54
Q5,1.32,7.54,0.11,0.56,1.35,0.65,0.46
Q5-Q1,0.49,5.79,0.18,0.71,0.31,0.06,0.29
Q5-No R&D,0.37,5.09,-0.04,-0.19,0.42,0.14,0.25
CRSP VW Benchmark,—,—,—,—,—,—,0.42


Portfolio,Mean Return (%),Std Dev (%),Alpha (monthly %),t-statistic,Beta (Mkt),Beta (SMB),Beta (HML),R²,Sharpe
No R&D,1.08,5.81,-0.09,-1.01,1.03,0.77,0.41,0.87,0.45
Q1,0.6,6.21,-0.48***,-6.73,1.04,0.85,-0.16,0.93,0.15
Q2,0.99,6.38,-0.12,-1.61,1.06,0.95,-0.09,0.93,0.36
Q3,1.03,6.68,-0.10,-1.06,1.08,1.01,-0.06,0.9,0.37
Q4,1.39,7.45,0.23*,1.68,1.11,1.19,-0.04,0.83,0.5
Q5,2.34,9.81,1.11***,4.2,1.19,1.52,-0.08,0.64,0.71
Q5-Q1,1.74,5.98,1.58***,6.35,0.14,0.67,0.08,0.13,1.01
Q5-No R&D,1.26,5.97,1.20***,5.36,0.16,0.75,-0.49,0.3,0.73
CRSP VW Benchmark,—,—,—,—,—,—,—,—,0.42


Portfolio,Mean Return (%),Std Dev (%),Alpha (monthly %),t-statistic,Beta (Mkt),Beta (SMB),Beta (HML),R²,Sharpe
No R&D,0.94,4.37,-0.07,-1.2,0.94,-0.0,0.15,0.92,0.49
Q1,0.83,5.17,-0.11,-1.2,0.99,-0.05,-0.33,0.85,0.34
Q2,1.2,5.38,0.20**,2.1,1.03,0.07,-0.23,0.85,0.56
Q3,1.17,5.54,0.12,1.22,1.06,0.17,-0.13,0.84,0.53
Q4,1.29,6.22,0.16,1.15,1.14,0.23,-0.01,0.75,0.54
Q5,1.32,7.54,-0.02,-0.13,1.32,0.47,0.34,0.69,0.46
Q5-Q1,0.49,5.79,0.08,0.36,0.33,0.51,0.67,0.22,0.29
Q5-No R&D,0.37,5.09,0.04,0.2,0.38,0.47,0.19,0.21,0.25
CRSP VW Benchmark,—,—,—,—,—,—,—,—,0.42


Portfolio,Mean Return (%),Std Dev (%),Alpha (monthly %),t-statistic,Beta,R²,Sharpe
No R&D,1.17,4.93,-0.16,-0.89,0.93,0.69,0.43
Q1,0.78,5.97,-0.75***,-3.87,1.18,0.76,0.13
Q2,1.11,5.89,-0.37*,-1.77,1.12,0.71,0.33
Q3,1.22,6.05,-0.28,-1.25,1.14,0.69,0.38
Q4,1.6,6.44,0.10,0.39,1.14,0.61,0.56
Q5,2.78,7.79,1.26***,3.27,1.17,0.44,0.99
Q5-Q1,2.0,4.09,2.03***,7.31,-0.02,0.0,1.69
Q5-No R&D,1.62,4.24,1.30***,4.64,0.23,0.05,1.32
CRSP VW Benchmark,—,—,—,—,—,—,0.42


...

Portfolio,Mean Return (%),Std Dev (%),Alpha (monthly %),t-statistic,Beta (Mkt),Beta (SMB),Beta (HML),R²,Sharpe
No R&D,1.31,4.31,-0.10,-1.49,0.96,0.09,0.1,0.95,0.61
Q1,1.28,5.31,-0.02,-0.12,0.96,-0.11,-0.41,0.84,0.47
Q2,1.57,4.78,0.22,1.58,0.94,0.01,-0.09,0.82,0.74
Q3,1.34,5.01,-0.08,-0.6,1.01,0.18,0.0,0.84,0.54
Q4,1.54,5.26,0.14,0.87,1.01,0.14,-0.08,0.79,0.65
Q5,1.69,5.94,0.08,0.4,1.16,0.35,0.25,0.74,0.66
Q5-Q1,0.4,4.52,0.10,0.35,0.2,0.46,0.66,0.15,0.31
Q5-No R&D,0.37,3.69,0.18,0.76,0.2,0.26,0.15,0.08,0.35
CRSP VW Benchmark,—,—,—,—,—,—,—,—,0.42


Portfolio,Mean Return (%),Std Dev (%),Alpha (monthly %),t-statistic,Beta,R²,Sharpe
No R&D,1.1,6.47,0.30,1.58,1.24,0.77,0.53
Q1,0.57,5.93,-0.20,-1.37,1.19,0.85,0.27
Q2,0.91,6.4,0.10,0.62,1.28,0.84,0.44
Q3,0.93,6.89,0.09,0.44,1.33,0.79,0.41
Q4,1.25,7.87,0.35,1.29,1.43,0.69,0.5
Q5,1.95,10.7,0.93**,1.98,1.65,0.5,0.6
Q5-Q1,1.39,6.9,1.08***,2.64,0.46,0.09,0.7
Q5-No R&D,0.86,6.45,0.58,1.52,0.41,0.09,0.46
CRSP VW Benchmark,—,—,—,—,—,—,0.42


Portfolio,Mean Return (%),Std Dev (%),Alpha (monthly %),t-statistic,Beta,R²,Sharpe
No R&D,0.68,4.43,0.06,0.79,0.93,0.92,0.45
Q1,0.56,4.7,-0.06,-0.55,0.94,0.84,0.34
Q2,0.88,5.72,0.13,0.96,1.15,0.85,0.47
Q3,1.02,5.83,0.27*,1.88,1.17,0.85,0.54
Q4,1.11,6.93,0.28,1.29,1.31,0.75,0.5
Q5,1.04,8.62,0.07,0.24,1.55,0.68,0.37
Q5-Q1,0.48,6.34,0.07,0.2,0.61,0.2,0.26
Q5-No R&D,0.36,5.82,-0.05,-0.17,0.63,0.24,0.21
CRSP VW Benchmark,—,—,—,—,—,—,0.42


Portfolio,Mean Return (%),Std Dev (%),Alpha (monthly %),t-statistic,Beta (Mkt),Beta (SMB),Beta (HML),R²,Sharpe
No R&D,1.1,6.47,0.14,1.05,1.07,0.87,0.38,0.9,0.53
Q1,0.57,5.93,-0.31***,-3.26,1.06,0.76,-0.12,0.94,0.27
Q2,0.91,6.4,-0.03,-0.27,1.13,0.87,-0.12,0.95,0.44
Q3,0.93,6.89,-0.05,-0.42,1.16,1.01,-0.14,0.91,0.41
Q4,1.25,7.87,0.19,0.93,1.21,1.26,-0.14,0.84,0.5
Q5,1.95,10.7,0.74*,1.85,1.37,1.66,-0.32,0.64,0.6
Q5-Q1,1.39,6.9,1.05***,2.72,0.31,0.89,-0.21,0.2,0.7
Q5-No R&D,0.86,6.45,0.60*,1.78,0.3,0.79,-0.7,0.29,0.46
CRSP VW Benchmark,—,—,—,—,—,—,—,—,0.42


Portfolio,Mean Return (%),Std Dev (%),Alpha (monthly %),t-statistic,Beta (Mkt),Beta (SMB),Beta (HML),R²,Sharpe
No R&D,0.68,4.43,0.01,0.11,0.91,0.04,0.13,0.92,0.45
Q1,0.56,4.7,-0.07,-0.67,0.96,-0.06,-0.27,0.88,0.34
Q2,0.88,5.72,0.11,0.94,1.16,0.06,-0.33,0.89,0.47
Q3,1.02,5.83,0.22*,1.71,1.16,0.15,-0.22,0.87,0.54
Q4,1.11,6.93,0.19,0.91,1.23,0.46,-0.06,0.77,0.5
Q5,1.04,8.62,-0.09,-0.32,1.4,0.83,0.21,0.74,0.37
Q5-Q1,0.48,6.34,-0.02,-0.06,0.44,0.89,0.48,0.36,0.26
Q5-No R&D,0.36,5.82,-0.10,-0.33,0.49,0.79,0.07,0.35,0.21
CRSP VW Benchmark,—,—,—,—,—,—,—,—,0.42


In [ ]:
# Transition plots: two charts (EW, VW), each with 3 subplots (Alpha, Sharpe, t-stat). Slide aspect ratio; direct labels at end of each line (no legend).
from plotly.subplots import make_subplots

# PPT slide proportions (16:9 widescreen): 1280 x 720 px
SLIDE_WIDTH, SLIDE_HEIGHT = 1280, 720

if len(SUBPERIODS) >= 2:
    p1_label, p1_start, p1_end = SUBPERIODS[0]
    p2_label, p2_start, p2_end = SUBPERIODS[1]
    pm_capm_1, pm_ff3_1 = metrics_for_period(portfolio_returns_norm, market_returns, rf_rate_monthly, p1_start, p1_end)
    pm_capm_2, pm_ff3_2 = metrics_for_period(portfolio_returns_norm, market_returns, rf_rate_monthly, p2_start, p2_end)

    def portfolio_name(p):
        return p.split('_', 1)[1] if '_' in str(p) else str(p)

    # Use FF3 for transition; same metrics for both weightings
    for weighting in ['Equal-Weighted', 'Value-Weighted']:
        pm1, pm2 = pm_ff3_1, pm_ff3_2
        if pm1 is None or pm2 is None:
            continue
        a1 = pm1[pm1['portfolio'].str.startswith(weighting, na=False)].copy()
        a2 = pm2[pm2['portfolio'].str.startswith(weighting, na=False)].copy()
        a1['Portfolio'] = a1['portfolio'].apply(portfolio_name)
        a2['Portfolio'] = a2['portfolio'].apply(portfolio_name)
        port_order = ['No R&D', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q5-Q1', 'Q5-No R&D']
        common = set(a1['Portfolio'].tolist()) & set(a2['Portfolio'].tolist())
        ports = [p for p in port_order if p in common]
        if not ports:
            continue
        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=('Alpha (monthly %)', 'Sharpe ratio', 'Alpha t-statistic'),
            horizontal_spacing=0.08
        )
        for port in ports:
            r1, r2 = a1[a1['Portfolio'] == port], a2[a2['Portfolio'] == port]
            if r1.empty or r2.empty:
                continue
            alpha_1 = float(r1['alpha'].iloc[0] * 100)
            alpha_2 = float(r2['alpha'].iloc[0] * 100)
            sh_1 = float(r1['sharpe_ratio'].iloc[0])
            sh_2 = float(r2['sharpe_ratio'].iloc[0])
            t_1 = float(r1['alpha_tstat'].iloc[0])
            t_2 = float(r2['alpha_tstat'].iloc[0])
            c = colors.get(port, '#888')
            fig.add_trace(
                go.Scatter(x=[p1_label, p2_label], y=[alpha_1, alpha_2], mode='lines+markers', name=port,
                           line=dict(color=c, width=2), marker=dict(size=8), showlegend=False),
                row=1, col=1)
            fig.add_trace(
                go.Scatter(x=[p1_label, p2_label], y=[sh_1, sh_2], mode='lines+markers', name=port,
                           line=dict(color=c, width=2), marker=dict(size=8), showlegend=False),
                row=1, col=2)
            fig.add_trace(
                go.Scatter(x=[p1_label, p2_label], y=[t_1, t_2], mode='lines+markers', name=port,
                           line=dict(color=c, width=2), marker=dict(size=8), showlegend=False),
                row=1, col=3)
            # Direct labels at end of each line (proximate to line, like the voltage chart)
            fig.add_annotation(x=p2_label, y=alpha_2, text=port, xref='x1', yref='y1', xanchor='left', xshift=10, showarrow=False, font=dict(color=c, size=10))
            fig.add_annotation(x=p2_label, y=sh_2, text=port, xref='x2', yref='y2', xanchor='left', xshift=10, showarrow=False, font=dict(color=c, size=10))
            fig.add_annotation(x=p2_label, y=t_2, text=port, xref='x3', yref='y3', xanchor='left', xshift=10, showarrow=False, font=dict(color=c, size=10))
        fig.update_layout(
            title=dict(text=f'Transition by period — {weighting}', x=0.5, xanchor='center'),
            template='plotly_dark',
            width=SLIDE_WIDTH,
            height=SLIDE_HEIGHT,
            showlegend=False,
            margin=dict(l=80, r=120, t=80, b=60),
        )
        fig.update_xaxes(title_text='Period', row=1, col=1)
        fig.update_xaxes(title_text='Period', row=1, col=2)
        fig.update_xaxes(title_text='Period', row=1, col=3)
        fig.update_yaxes(title_text='Alpha (monthly %)', row=1, col=1)
        fig.update_yaxes(title_text='Sharpe', row=1, col=2)
        fig.update_yaxes(title_text='t-stat', row=1, col=3)
        fig.show()
else:
    print('Need at least two subperiods for transition charts.')

In [ ]:
# Results table for report: long-short portfolios (Q5-No R&D and Q5-Q1), subperiods, CAPM and FF3F
GROUP_NAME = "Darius Rio Zac"
PORTFOLIO_MAP = {
    'Equal-Weighted_Q5-No R&D': 'EW Q5-Q0',
    'Equal-Weighted_Q5-Q1': 'EW Q5-Q1',
    'Value-Weighted_Q5-No R&D': 'VW Q5-Q0',
    'Value-Weighted_Q5-Q1': 'VW Q5-Q1',
}
LONG_SHORT_PORTFOLIOS = list(PORTFOLIO_MAP.keys())
PORTFOLIO_ORDER = ['EW Q5-Q0', 'EW Q5-Q1', 'VW Q5-Q0', 'VW Q5-Q1']
REGRESSION_ORDER = ['CAPM', 'FF3F']

def one_model_rows(pm, period_label, model_name):
    """Extract table rows from a portfolio-metrics DataFrame (CAPM or FF3) for long-short portfolios only. Returns a DataFrame."""
    if pm is None or pm.empty or 'portfolio' not in pm.columns:
        return pd.DataFrame()
    subset = pm[pm['portfolio'].isin(LONG_SHORT_PORTFOLIOS)].copy()
    subset['Portfolio'] = subset['portfolio'].map(PORTFOLIO_MAP)
    subset['Time Period'] = period_label
    subset['Regression (CAPM/FF3F)'] = model_name
    subset['Group'] = GROUP_NAME
    subset['Alpha'] = subset['alpha']
    subset['Alpha t stat'] = subset['alpha_tstat']
    subset['mkt b'] = subset['beta']
    if model_name == 'CAPM':
        subset['smb b'] = np.nan
        subset['hml b'] = np.nan
    else:
        subset['smb b'] = subset['beta_smb']
        subset['hml b'] = subset['beta_hml']
    subset['r^2'] = subset['r_squared']
    subset['sharpe'] = subset['sharpe_ratio']
    return subset

row_dfs = []
for period_label, start_date, end_date in SUBPERIODS:
    pm_capm, pm_ff3 = metrics_for_period(portfolio_returns_norm, market_returns, rf_rate_monthly, start_date, end_date)
    for pm, model in [(pm_capm, 'CAPM'), (pm_ff3, 'FF3F')]:
        df = one_model_rows(pm, period_label, model)
        if df is not None and len(df) > 0:
            row_dfs.append(df)

if not row_dfs:
    results_table = pd.DataFrame(columns=['Group', 'Portfolio', 'Time Period', 'Regression (CAPM/FF3F)', 'Alpha', 'Alpha t stat', 'mkt b', 'smb b', 'hml b', 'r^2', 'sharpe'])
else:
    results_table = pd.concat(row_dfs, ignore_index=True)[['Group', 'Portfolio', 'Time Period', 'Regression (CAPM/FF3F)', 'Alpha', 'Alpha t stat', 'mkt b', 'smb b', 'hml b', 'r^2', 'sharpe']]
    results_table['Portfolio'] = pd.Categorical(results_table['Portfolio'], categories=PORTFOLIO_ORDER, ordered=True)
    results_table['Regression (CAPM/FF3F)'] = pd.Categorical(results_table['Regression (CAPM/FF3F)'], categories=REGRESSION_ORDER, ordered=True)
    results_table = results_table.sort_values(['Time Period', 'Portfolio', 'Regression (CAPM/FF3F)']).reset_index(drop=True)
display(results_table)

,Group,Portfolio,Time Period,Regression (CAPM/FF3F),Alpha,Alpha t stat,mkt b,smb b,hml b,r^2,sharpe
0,Darius Rio Zac,EW Q5-Q0,1980-2000,CAPM,0.013026,4.643248,0.226046,NaN,NaN,0.054471,1.318903
1,Darius Rio Zac,EW Q5-Q0,1980-2000,FF3F,0.016417,6.669173,0.087238,0.679016,-0.214798,0.270248,1.318903
2,Darius Rio Zac,EW Q5-Q1,1980-2000,CAPM,0.020303,7.306069,-0.023179,NaN,NaN,0.000617,1.693110
3,Darius Rio Zac,EW Q5-Q1,1980-2000,FF3F,0.019945,7.840031,-0.010263,0.633314,0.243205,0.160477,1.693110
4,Darius Rio Zac,VW Q5-Q0,1980-2000,CAPM,0.001220,0.497583,0.182241,NaN,NaN,0.046704,0.350977
5,Darius Rio Zac,VW Q5-Q0,1980-2000,FF3F,0.001837,0.764541,0.200018,0.255861,0.145099,0.082914,0.350977
6,Darius Rio Zac,VW Q5-Q1,1980-2000,CAPM,0.003388,1.102385,0.047156,NaN,NaN,0.002085,0.309354
7,Darius Rio Zac,VW Q5-Q1,1980-2000,FF3F,0.000995,0.350736,0.204825,0.459820,0.658063,0.148280,0.309354
8,Darius Rio Zac,EW Q5-Q0,2001-2022,CAPM,0.005835,1.517092,0.412086,NaN,NaN,0.085435,0.459411
9,Darius Rio Zac,EW Q5-Q0,2001-2022,FF3F,0.006047,1.778253,0.301263,0.788511,-0.701717,0.289426,0.459411


In [ ]:
results_table.to_clipboard()

In [ ]:
db.close()